In [ ]:
"""
Aula 04: Lógica Proposicional — Conectivos e Blocos de Permissivos
Sistema Automatizado de Envasamento, Tampagem e Inspeção de Garrafas
"""

from typing import Dict, Any


def xor(a: bool, b: bool) -> bool:
    """Implementação formal do conectivo de Disjunção Exclusiva (XOR)."""
    return bool(a ^ b)


def avaliar_planta(estado: Dict[str, Any]) -> Dict[str, Any]:
    """
    Avalia os permissivos e intertravamentos (trips) da planta de envase
    com base no vetor de estados dos sensores e atuadores.
    """
    # 1. Modos de Operação e Segurança Geral
    modo_valido = xor(estado["modo_auto"], estado["modo_manual"])
    emergencia_ativa = estado["e_stop"]
    sem_emergencia = not emergencia_ativa

    # 2. Permissivo e Trip da Bomba Centrífuga BC1
    p_bc1_base = (
        estado["sp1_ok"]
        and estado["sq1_ok"]
        and estado["ls_vs1_open"]
        and not estado["p_as1_high"]
        and sem_emergencia
    )
    permissivo_bc1 = p_bc1_base and modo_valido
    trip_bc1 = (
        (not estado["sp1_ok"])
        or (not estado["sq1_ok"])
        or (not estado["ls_vs1_open"])
        or estado["p_as1_high"]
        or emergencia_ativa
    )

    # 3. Permissivo e Trip da Esteira Transportadora RC1
    atuadores_estendidos = (
        estado["act_vs2"]
        or estado["ext_vs3"]
        or estado["ext_vs4"]
        or estado["ext_vs5"]
    )
    permissivo_rc1 = (not atuadores_estendidos) and sem_emergencia and modo_valido
    trip_rc1 = emergencia_ativa or atuadores_estendidos

    # 4. Permissivo da Válvula de Enchimento VS2
    permissivo_vs2 = (
        estado["pos_garrafa_vs2"]
        and (not estado["cmd_rc1"])
        and estado["sp2_ok"]
        and sem_emergencia
        and modo_valido
    )

    # 5. Permissivo da Inspeção de Nível VS3 e Avaliação de Nível
    permissivo_vs3 = (
        estado["pos_garrafa_vs3"]
        and (not estado["cmd_rc1"])
        and sem_emergencia
        and modo_valido
    )
    aprovacao_nivel = estado["ext_vs3"] and estado["sl1_nivel_ok"]

    # 6. Permissivo de Tampagem VS4 (AC1)
    permissivo_vs4 = (
        estado["pos_garrafa_vs4"]
        and estado["nivel_garrafa_aprovado"]
        and (not estado["cmd_rc1"])
        and sem_emergencia
        and modo_valido
    )

    # 7. Permissivo da Inspeção de Vedação VS5 e Avaliação de Vedação
    permissivo_vs5 = (
        estado["pos_garrafa_vs5"]
        and (not estado["cmd_rc1"])
        and sem_emergencia
        and modo_valido
    )
    aprovacao_vedacao = estado["ext_vs5"] and estado["sfc1_tampa_detectada"]

    return {
        "Modo_Valido": modo_valido,
        "P_BC1 (Bomba)": permissivo_bc1,
        "Trip_BC1": trip_bc1,
        "P_RC1 (Esteira)": permissivo_rc1,
        "Trip_RC1": trip_rc1,
        "P_VS2 (Envase)": permissivo_vs2,
        "P_VS3 (Insp. Nível)": permissivo_vs3,
        "Aprov_Nivel": aprovacao_nivel,
        "P_VS4 (Tampagem)": permissivo_vs4,
        "P_VS5 (Insp. Vedação)": permissivo_vs5,
        "Aprov_Vedacao": aprovacao_vedacao,
    }


# ==============================================================================
# Execução e Validação de Cenários Operacionais
# ==============================================================================
if __name__ == "__main__":
    cenarios = {
        "1. Operação Nominal de Envase (Pronto para Dosagem)": {
            "modo_auto": True,
            "modo_manual": False,
            "e_stop": False,
            "sp1_ok": True,
            "sq1_ok": True,
            "ls_vs1_open": True,
            "p_as1_high": False,
            "sp2_ok": True,
            "cmd_rc1": False,
            "act_vs2": False,
            "ext_vs3": False,
            "ext_vs4": False,
            "ext_vs5": False,
            "pos_garrafa_vs2": True,
            "pos_garrafa_vs3": False,
            "pos_garrafa_vs4": False,
            "pos_garrafa_vs5": False,
            "sl1_nivel_ok": False,
            "nivel_garrafa_aprovado": True,
            "sfc1_tampa_detectada": False,
        },
        "2. Falha de Sucção na Bomba (Tanque TS1 Vazio / sq1 = False)": {
            "modo_auto": True,
            "modo_manual": False,
            "e_stop": False,
            "sp1_ok": False,
            "sq1_ok": False,
            "ls_vs1_open": True,
            "p_as1_high": False,
            "sp2_ok": False,
            "cmd_rc1": False,
            "act_vs2": False,
            "ext_vs3": False,
            "ext_vs4": False,
            "ext_vs5": False,
            "pos_garrafa_vs2": True,
            "pos_garrafa_vs3": False,
            "pos_garrafa_vs4": False,
            "pos_garrafa_vs5": False,
            "sl1_nivel_ok": False,
            "nivel_garrafa_aprovado": False,
            "sfc1_tampa_detectada": False,
        },
        "3. Emergência Acionada (E-Stop = True)": {
            "modo_auto": True,
            "modo_manual": False,
            "e_stop": True,
            "sp1_ok": True,
            "sq1_ok": True,
            "ls_vs1_open": True,
            "p_as1_high": False,
            "sp2_ok": True,
            "cmd_rc1": False,
            "act_vs2": False,
            "ext_vs3": False,
            "ext_vs4": False,
            "ext_vs5": False,
            "pos_garrafa_vs2": True,
            "pos_garrafa_vs3": False,
            "pos_garrafa_vs4": False,
            "pos_garrafa_vs5": False,
            "sl1_nivel_ok": False,
            "nivel_garrafa_aprovado": True,
            "sfc1_tampa_detectada": False,
        },
        "4. Conflito: Tentativa de Movimento com Atuador de Tampagem Estendido": {
            "modo_auto": True,
            "modo_manual": False,
            "e_stop": False,
            "sp1_ok": True,
            "sq1_ok": True,
            "ls_vs1_open": True,
            "p_as1_high": False,
            "sp2_ok": True,
            "cmd_rc1": True,
            "act_vs2": False,
            "ext_vs3": False,
            "ext_vs4": True,  # Atuador VS4/AC1 estendido na garrafa
            "ext_vs5": False,
            "pos_garrafa_vs2": False,
            "pos_garrafa_vs3": False,
            "pos_garrafa_vs4": True,
            "pos_garrafa_vs5": False,
            "sl1_nivel_ok": False,
            "nivel_garrafa_aprovado": True,
            "sfc1_tampa_detectada": False,
        },
    }

    print("=" * 80)
    print("AVALIAÇÃO DOS BLOCOS DE PERMISSIVOS E INTERTRAVAMENTOS (CLP)")
    print("=" * 80)
    for nome_cenario, estado in cenarios.items():
        print(f"\n>>> Cenário: {nome_cenario}")
        resultados = avaliar_planta(estado)
        for chave, valor in resultados.items():
            status = "✅ OK / ATIVO" if valor else "❌ NÃO ATENDIDO / INATIVO"
            print(f"  • {chave:<25}: {str(valor):<6} | {status}")